# Disease Classification - PlantVillage MobileNetV2 Fine-Tuning

This notebook trains a MobileNetV2-based classifier on the PlantVillage dataset
for crop disease detection. The trained model is exported as a quantized TFLite
model compatible with the Coral Edge TPU on the AgroBot rover.

**Hardware requirements:** Google Colab with T4 GPU runtime.

**Expected output:** `disease_model_float16.tflite` and `disease_model_quant.tflite`
for deployment to `models/` on the Raspberry Pi.

In [ ]:
# Install dependencies
!pip install -q tensorflow tensorflow-datasets matplotlib scikit-learn

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load PlantVillage dataset from tensorflow_datasets
# The dataset contains 54,000+ images across 38 disease classes (14 crops)
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    'plant_village',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

NUM_CLASSES = ds_info.features['label'].num_classes
CLASS_NAMES = ds_info.features['label'].names
print(f"Number of classes: {NUM_CLASSES}")
print(f"Training samples: {len(ds_train)}")
print(f"Validation samples: {len(ds_val)}")
print(f"Test samples: {len(ds_test)}")

In [ ]:
# Data exploration and visualization
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, (image, label) in enumerate(ds_train.take(12)):
    ax = axes[i // 4, i % 4]
    ax.imshow(image.numpy())
    ax.set_title(CLASS_NAMES[label.numpy()], fontsize=8)
    ax.axis('off')
plt.suptitle('PlantVillage Dataset Samples', fontsize=14)
plt.tight_layout()
plt.show()

# Class distribution
label_counts = {}
for _, label in ds_train:
    lbl = label.numpy()
    label_counts[lbl] = label_counts.get(lbl, 0) + 1

plt.figure(figsize=(14, 5))
plt.bar(range(NUM_CLASSES), [label_counts.get(i, 0) for i in range(NUM_CLASSES)])
plt.xlabel('Class Index')
plt.ylabel('Count')
plt.title('Class Distribution in Training Set')
plt.show()

In [ ]:
# Data augmentation pipeline
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])


def preprocess_train(image, label):
    """Resize, normalize, and augment training images."""
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    image = data_augmentation(image, training=True)
    return image, label


def preprocess_eval(image, label):
    """Resize and normalize evaluation images (no augmentation)."""
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


# Build data pipelines
train_ds = (
    ds_train
    .shuffle(1000)
    .map(preprocess_train, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    ds_val
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    ds_test
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"Train batches: {len(train_ds)}")
print(f"Val batches: {len(val_ds)}")
print(f"Test batches: {len(test_ds)}")

In [ ]:
# Build MobileNetV2 transfer learning model
# Freeze the base model initially for feature extraction
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

# Custom classification head
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# Training with callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_disease_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
    ),
]

# Phase 1: Train with frozen base (feature extraction)
print("Phase 1: Feature extraction (frozen base)")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)

# Phase 2: Fine-tune top layers of base model
print("\nPhase 2: Fine-tuning top layers")
base_model.trainable = True
# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
)

In [ ]:
# Evaluate with confusion matrix and classification report
# Load best model
model = tf.keras.models.load_model('best_disease_model.keras')

# Get predictions on test set
y_true = []
y_pred = []
for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Overall accuracy
accuracy = np.mean(y_true == y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Disease Classification')
plt.tight_layout()
plt.show()

# Training history plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_phase2.history['accuracy'], label='Train')
ax1.plot(history_phase2.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()
ax2.plot(history_phase2.history['loss'], label='Train')
ax2.plot(history_phase2.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TFLite export - Float16 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_float16 = converter.convert()

os.makedirs('output', exist_ok=True)
with open('output/disease_model_float16.tflite', 'wb') as f:
    f.write(tflite_float16)
print(f"Float16 model size: {len(tflite_float16) / 1024 / 1024:.2f} MB")

# TFLite export - Full INT8 quantization (for Edge TPU)
def representative_dataset():
    """Generate representative dataset for INT8 calibration."""
    for images, _ in test_ds.take(100):
        for i in range(images.shape[0]):
            yield [tf.expand_dims(images[i], axis=0)]


converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()

with open('output/disease_model_quant.tflite', 'wb') as f:
    f.write(tflite_int8)
print(f"INT8 model size: {len(tflite_int8) / 1024 / 1024:.2f} MB")

In [ ]:
# Edge TPU compilation
# Install the Edge TPU compiler
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update && sudo apt-get install -y edgetpu-compiler

# Compile the INT8 quantized model for Edge TPU
!edgetpu_compiler -s -o output/ output/disease_model_quant.tflite

print("\nCompiled models:")
!ls -la output/

In [ ]:
# Copy to models/ directory on the Pi
# After downloading from Colab, place files as follows:
#
#   models/disease_model_quant_edgetpu.tflite  -> Edge TPU model
#   models/disease_model_float16.tflite        -> CPU fallback model
#
# These are loaded by pi/ai/disease_detection.py:
#   DiseaseClassifier(model_path='models/disease_model_quant_edgetpu.tflite')
#
# To download from Colab:
from google.colab import files
files.download('output/disease_model_quant_edgetpu.tflite')
files.download('output/disease_model_float16.tflite')

print("Done! Transfer these files to the Pi's models/ directory.")